# Day 40 · Webhook 与索引同步

**配套讲义**: [`days/day-40.md`](../days/day-40.md) ｜ **本地可跑，不需要 GPU**

订阅 `products/update` / `orders/create` / `refunds/create`；商品变更时**增量**更新向量索引，验收标准是「改一个商品标题，30 秒内 RAG 索引同步」。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w7.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys
print("python:", sys.version.split()[0])
for m in ("numpy", "PIL", "yaml", "pandas"):
    try:
        mod = __import__(m)
        print(f"  {m:7s} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"  {m:7s} ❌ 缺 → pip install {m}")
print("\n→ 本机没 GPU 不影响今天：今天只用纯 Python / numpy")

## 1. 自检

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.shopify.webhooks"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout or r.stderr)

## 2. 幂等存储：亲手验一遍

In [ ]:
import sys; sys.path.insert(0, "..")
from src.shopify.webhooks import IdempotencyStore

store = IdempotencyStore()
wid = "wh_abc123"
print("第一次:", store.seen(wid), "（应 False —— 没见过）")
print("第二次:", store.seen(wid), "（应 True —— 已处理过，跳过）")
print("\n→ 如果这里返回 False，你会重复处理同一个事件")

## 3. 增量 vs 全量的成本对比

In [ ]:
import time

n_products, n_images_per = 1000, 2

print("全量重建：")
print(f"  下载 {n_products * n_images_per} 张图 + 编码 → 约 "
      f"{n_products * n_images_per * 0.5 / 60:.1f} 分钟")

print("\n增量（改 1 个商品标题）：")
print(f"  只需重编码 {n_images_per} 张图 + 更新 1 条文本 → 约 1 秒")
print("\n→ 差 3 个数量级。这就是为什么必须做增量。")

## 验收清单

- [ ] `python -m src.shopify.webhooks` 自检通过（HMAC base64 + 幂等）
- [ ] **改一个商品标题，30 秒内索引同步**（这是硬验收）
- [ ] 能说清 webhook 的 HMAC 验证和幂等处理（各自防什么）
- [ ] 商品下架时索引被正确删除（不会搜到已下架商品）

**卡住了？** 回看 [`days/day-40.md`](../days/day-40.md) 第五节「容易踩的坑」。

> **明天**：`days/day-41.md` —— 计费与合规（含 GDPR webhook）